# Деление Томаса в GInv

In [1]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath('..'))

from ginv.ginv import *

In [2]:
def variables(names):
    Monom.init(names)
    Monom.cmp = Poly.cmp = Monom.TOPdeglex
    return [Poly(Monom(i)) for i in range(len(names))]

def compute(polys, division):
    Wrap.id = 0
    idiv = division()
    start = time.process_time()
    ginvBlockLow(polys, idiv)
    return idiv, time.process_time() - start

## Мономиальный идеал ⟨x², y²⟩

In [3]:
x, y = variables(['x', 'y'])

janet, _ = compute([x**2, y**2], Janet)
thomas, _ = compute([x**2, y**2], Thomas)

print('Жане: ', sorted(str(w.lm) for w in janet))
print('Томас:', sorted(str(w.lm) for w in thomas))

Жане:  ['x**2', 'x*y**2', 'y**2']
Томас: ['x**2', 'x**2*y', 'x**2*y**2', 'x*y**2', 'y**2']


Переменная мультипликативна, если её степень равна максимальной степени `h` в базисе.

In [4]:
print('h =', thomas.h)
for w in thomas:
    print(f'{str(w.lm):10s}', [v for v, nm in zip('xy', w.nonmult) if not nm])

h = [2, 2]
y**2       ['y']
x**2       ['x']
x*y**2     ['y']
x**2*y     ['x']
x**2*y**2  ['x', 'y']


Деление Томаса не зависит от порядка переменных, поэтому базис Томаса переходит в себя при x ↔ y.

In [5]:
def symmetric(idiv):
    lms = {tuple(w.lm) for w in idiv}
    return lms == {m[::-1] for m in lms}

print('Жане: ', symmetric(janet))
print('Томас:', symmetric(thomas))

Жане:  False
Томас: True


## Тестовые системы

Для каждой системы: размер инволютивного базиса и время, совпадение базисов Грёбнера и проверка, что все немультипликативные продолжения базиса Томаса редуцируются к нулю.

In [6]:
def load(name):
    with open(f'../json/{name}.json', encoding='utf-8') as f:
        data = json.load(f)
    gens = dict(zip(data['variables'], variables(data['variables'])))
    return [eval(e.replace('^', '**'), dict(gens)) for e in data['equations']]

def groebner(idiv):
    return {frozenset((tuple(m), c / w.poly.lc()) for m, c in w.poly) for w in idiv.GB()}

def involutive(idiv):
    for w in idiv:
        for i, nm in enumerate(w.nonmult):
            if nm:
                p = w.poly.prolong(i)
                p.NFhead(idiv)
                if p:
                    return False
    return True

In [7]:
systems = ['quadfor2', 'solotarev', 'conform1', 'chandra4', 'lorentz', 'hunecke', 'quadgrid']

print(f"{'':10s} {'Жане':>6s} {'Томас':>6s} {'t Жане':>8s} {'t Томас':>8s}   ГБ   ГБ равны   инволютивен")
for name in systems:
    janet, tj = compute(load(name), Janet)
    thomas, tt = compute(load(name), Thomas)
    gb_janet, gb_thomas = groebner(janet), groebner(thomas)
    print(f'{name:10s} {janet.count():6d} {thomas.count():6d} {tj:7.2f}с {tt:7.2f}с {len(gb_thomas):4d}   '
          f'{str(gb_janet == gb_thomas):9s}  {involutive(thomas)}')

             Жане  Томас   t Жане  t Томас   ГБ   ГБ равны   инволютивен
quadfor2        4     22    0.00с    0.03с    4   True       True


solotarev      10     75    0.03с    0.30с    9   True       True


conform1       10     80    0.05с    0.48с   10   True       True


chandra4        8    112    0.05с    1.42с    8   True       True


lorentz        13    229    0.08с    1.52с   12   True       True


hunecke        15    312    0.00с    1.38с    9   True       True


quadgrid        8    139    0.11с    2.53с    8   True       True


## Базис Томаса целиком

In [10]:
def poly_str(p):
    s = ''
    for m, c in p:
        mono = str(m).replace('**', '^')
        if m.degree() == 0:
            term = str(abs(c))
        elif abs(c) == 1:
            term = mono
        else:
            term = f'{abs(c)}*{mono}'
        if s:
            s += (' - ' if c < 0 else ' + ') + term
        else:
            s = ('-' if c < 0 else '') + term
    return s

name = 'mickey'
polys = load(name)
for p in polys:
    print(poly_str(p))

thomas, _ = compute(polys, Thomas)

x^2 + 4*y^2 - 4
2*y^2 - x


`*` отмечает элементы базиса Грёбнера, справа указаны мультипликативные переменные.

In [11]:
print('h =', thomas.h)
for w in sorted(thomas):
    mult = ', '.join(str(Monom(i)) for i, nm in enumerate(w.nonmult) if not nm)
    print(f"{'*' if w.isGB() else ' '} {poly_str(w.poly):40s} {mult or '—'}")

h = [2, 2]
* 2*y^2 - x                                y
* x^2 + 2*x - 4                            x
  x*y^2 + x - 2                            y
  x^2*y + 2*x*y - 4*y                      x
  x^2*y^2 - 4*x + 4                        x, y
